# RT-DETRv2 (r50vd) — SportsMOT player detector · Kaggle trainer

Fine-tunes [`PekingU/rtdetr_v2_r50vd`](https://huggingface.co/PekingU/rtdetr_v2_r50vd)
into a single-class **player** detector on
[`Lekim89/sportsmot`](https://huggingface.co/datasets/Lekim89/sportsmot),
then pushes the trained model to
`smallTech/rtdetrv2-r50vd-sportsmot-players` on the Hugging Face Hub.

This is the **Kaggle free-tier** variant of `trainer.py`. It differs from the
Hugging Face Jobs version only in configuration needed to run on Kaggle's free
**T4** GPU (see the commented differences below).

## Before you run
In the Kaggle notebook's right-hand panel:
1. **Session options → Accelerator → `GPU T4`** (or `GPU T4 x2`).
2. **Session options → Internet → ON** — required to download the dataset (and,
   in interactive mode, to push the trained model back to the Hub).
3. **Add Data → your private `external-secrets` dataset** — the notebook reads the
   `HF_TOKEN` from it to push the model to the Hub. Create that dataset once on
   Kaggle: a private dataset named `external-secrets` containing a `secrets` file
   with `HF_TOKEN=<write token>`. If it isn't attached, the model is saved to the
   kernel output instead (see below).

Then **Run All**. Training takes roughly 3–5 hours on a T4. For an unattended run
that survives closing the browser tab, use **Save Version → Save & Run All (Commit)**.

**Headless / CLI runs:** this same notebook is launched non-interactively by the
workspace-root `kaggle_train.sh`, which references the `external-secrets` dataset
in the kernel metadata (this survives new versions, unlike Kaggle Secrets). The
notebook reads `HF_TOKEN` from it and pushes the model to the Hub. If the dataset
isn't attached, the model is saved to `/kaggle/working/model` instead.

In [ ]:
# --- 1. Dependencies -------------------------------------------------------
# CRITICAL on Kaggle: do NOT reinstall or upgrade torch. Kaggle's torch is built
# for the GPUs it provisions; replacing it with a PyPI wheel causes
#   "CUDA error: no kernel image is available for execution on the device"
# because the new wheel has no compiled kernels for this GPU's compute
# capability. That is exactly what broke an earlier run: installing
# `transformers[torch]` dragged in a torch that didn't match the GPU.
#
# Kaggle already ships torch, transformers (with RT-DETRv2 support), accelerate
# and huggingface_hub. We install ONLY the lightweight, torch-independent extras,
# and pass a pip constraint that pins the working torch build so nothing — not
# even a transitive dependency — can swap it out.
import subprocess, sys, tempfile
import torch

_con = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
_con.write(f"torch=={torch.__version__}\n")   # freeze the exact working build
_con.close()

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", _con.name,
     "albumentations>=1.4.10", "pycocotools", "torchmetrics"],
    check=True,
)

In [ ]:
# --- 2. Authentication -----------------------------------------------------
# The Hugging Face token comes from a PRIVATE Kaggle dataset named
# "external-secrets", mounted read-only at /kaggle/input/external-secrets/secrets
# (a KEY=VALUE file). Referencing that dataset in kernel-metadata.json survives
# new notebook versions — unlike Kaggle Secrets, which are dropped when the CLI
# pushes a new version. The dataset holds a `secrets` file of KEY=VALUE lines
# (e.g. HF_TOKEN=hf_...), maintained on Kaggle directly.
# If the dataset isn't attached, we fall back to saving the model to the output.
# huggingface_hub automatically uses HF_TOKEN from the environment when present.
import os

SECRETS_FILE = "/kaggle/input/external-secrets/secrets"

def _load_env_file(path):
    env = {}
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            env[key.strip()] = val.strip().strip('"').strip("'")
    return env

try:
    os.environ["HF_TOKEN"] = _load_env_file(SECRETS_FILE)["HF_TOKEN"]
    print("HF_TOKEN loaded from the external-secrets dataset -> will push to the Hub.")
except Exception as e:
    print(f"No HF_TOKEN from {SECRETS_FILE} ({type(e).__name__}) -> "
          "will save the model to /kaggle/working instead.")

In [ ]:
# --- 3. Imports and configuration -----------------------------------------
import random
from collections import defaultdict
from pathlib import Path

import albumentations as A          # bounding-box-aware image augmentation
import numpy as np
import torch
from huggingface_hub import HfApi, snapshot_download
from PIL import Image
from torch.utils.data import Dataset
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    Trainer,
    TrainingArguments,
)

DATASET_ID = "Lekim89/sportsmot"                 # MOTChallenge-format sports clips
CHECKPOINT = "PekingU/rtdetr_v2_r50vd"           # pretrained RT-DETRv2 backbone
HUB_MODEL_ID = "smallTech/rtdetrv2-r50vd-sportsmot-players"  # upload target
VAL_SEQ = "v_-6Os86HzwCs_c009"                   # held-out BASKETBALL sequence
TRAIN_STRIDE = 2                                 # adjacent frames are near-duplicates
VAL_STRIDE = 5
IMAGE_SIZE = 640
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

# KAGGLE DIFFERENCE #1 vs trainer.py:
# The T4 is a Turing GPU that supports fp16 but NOT bf16 (bf16 needs Ampere+).
# We auto-detect so the exact same cell also works on an A10G/A100 later.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
print(f"CUDA={torch.cuda.is_available()} bf16={USE_BF16} fp16={USE_FP16} "
      f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# --- Verbose logging (for debugging) ---------------------------------------
# Timestamped INFO logs from our own code and from transformers, plus a snapshot
# of versions and GPU. If a run breaks, the kernel log shows exactly where and in
# what environment. Bump level to logging.DEBUG for even more detail.
import logging
import transformers

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,                       # override Kaggle's default handler
)
logger = logging.getLogger("trainer")

transformers.utils.logging.set_verbosity_info()
transformers.utils.logging.enable_default_handler()
transformers.utils.logging.enable_explicit_format()

logger.info("torch=%s transformers=%s", torch.__version__, transformers.__version__)
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    logger.info("GPU=%s mem=%.1fGB cuda=%s bf16=%s fp16=%s",
                _p.name, _p.total_memory / 1e9, torch.version.cuda, USE_BF16, USE_FP16)
    # Guard against the "no kernel image is available" failure: verify torch was
    # actually built with kernels for THIS GPU's compute capability.
    _cap = torch.cuda.get_device_capability(0)          # e.g. (7, 5) for a T4
    _archs = torch.cuda.get_arch_list()                 # e.g. ['sm_70', 'sm_75', ...]
    logger.info("compute capability sm_%d%d | torch built for %s", _cap[0], _cap[1], _archs)
    if f"sm_{_cap[0]}{_cap[1]}" not in _archs:
        logger.error("torch has NO kernels for this GPU (sm_%d%d) -> forward passes will "
                     "fail with 'no kernel image is available'. Do NOT reinstall torch on Kaggle.",
                     _cap[0], _cap[1])
else:
    logger.warning("No CUDA GPU visible — training would be extremely slow.")

In [ ]:
# --- 4. Parsing the MOTChallenge annotations -------------------------------
# Each sequence has a gt/gt.txt with one line per box:
#   frame, track_id, x, y, w, h, conf, class, visibility
# We only need per-frame player boxes for detection, so we drop the track_id and
# keep (x, y, w, h) in pixel COCO format. conf==0 marks ignore boxes -> skipped.
def parse_gt(gt_path: Path) -> dict:
    """Return {frame_number: [[x, y, w, h], ...]}."""
    frames = defaultdict(list)
    for line in gt_path.read_text().strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        frame = int(parts[0])
        x, y, w, h = (float(v) for v in parts[2:6])
        conf = int(parts[6])
        if conf == 0 or w <= 0 or h <= 0:
            continue
        frames[frame].append([x, y, w, h])
    return frames


def load_samples(root: Path):
    """Walk the train sequences and build (image_path, boxes) lists.

    One sequence (VAL_SEQ, a basketball clip) is held out for validation so the
    reported metrics reflect the target basketball domain; the rest are training.
    Frames are subsampled by *_STRIDE because consecutive frames barely differ.
    """
    train_samples, val_samples = [], []
    for seq_dir in sorted((root / "train").iterdir()):
        if not seq_dir.is_dir():
            continue
        gt = parse_gt(seq_dir / "gt" / "gt.txt")
        img_dir = seq_dir / "img1"
        is_val = seq_dir.name == VAL_SEQ
        stride = VAL_STRIDE if is_val else TRAIN_STRIDE
        for i, img_path in enumerate(sorted(img_dir.glob("*.jpg"))):
            if i % stride != 0:
                continue
            frame = int(img_path.stem)
            boxes = gt.get(frame, [])
            if not boxes:                 # skip frames with no annotated players
                continue
            (val_samples if is_val else train_samples).append((img_path, boxes))
    return train_samples, val_samples

In [ ]:
# --- 5. Dataset + augmentation --------------------------------------------
# Light augmentation improves robustness to broadcast variation. albumentations
# keeps the bounding boxes in sync with the pixel transforms.
train_aug = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.GaussNoise(p=0.2),
    ],
    bbox_params=A.BboxParams(format="coco", label_fields=["labels"], clip=True, min_area=4),
)


class MotDataset(Dataset):
    """Yields the pixel_values + COCO-style labels that RT-DETRv2 expects."""

    def __init__(self, samples, processor, augment):
        self.samples = samples
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, boxes = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        labels = [0] * len(boxes)          # single class: 0 == "player"
        if self.augment:
            out = train_aug(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = out["image"], out["bboxes"], out["labels"]
        annotations = {
            "image_id": idx,
            "annotations": [
                {"bbox": list(b), "category_id": c, "area": b[2] * b[3], "iscrowd": 0}
                for b, c in zip(boxes, labels)
            ],
        }
        encoding = self.processor(images=image, annotations=annotations, return_tensors="pt")
        return {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "labels": encoding["labels"][0],
        }


def collate_fn(batch):
    # Detection targets have variable length, so labels stay a list (not stacked).
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": [b["labels"] for b in batch],
    }

In [ ]:
# --- 6. Download the annotated split and build the datasets ----------------
# Only the train/ split of SportsMOT carries ground-truth annotations, so that is
# all we pull (allow_patterns) rather than the full ~100k-file dataset.
print("Downloading annotated train split...")
root = Path(snapshot_download(DATASET_ID, repo_type="dataset", allow_patterns=["train/*"]))
train_samples, val_samples = load_samples(root)
print(f"train images: {len(train_samples)}, val images (basketball): {len(val_samples)}")

In [ ]:
# --- 7. Load the pretrained model and its image processor ------------------
# We replace the COCO head with a single "player" class. ignore_mismatched_sizes
# lets the new classification layer initialise fresh while reusing the backbone.
processor = AutoImageProcessor.from_pretrained(
    CHECKPOINT,
    do_resize=True,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE},
    use_fast=True,
)
model = AutoModelForObjectDetection.from_pretrained(
    CHECKPOINT,
    id2label={0: "player"},
    label2id={"player": 0},
    anchor_image_size=None,
    ignore_mismatched_sizes=True,
)

train_ds = MotDataset(train_samples, processor, augment=True)
val_ds = MotDataset(val_samples, processor, augment=False)

In [ ]:
# --- 8. Train --------------------------------------------------------------
# KAGGLE DIFFERENCE #2 vs trainer.py:
#   * batch size 4 (not 8) + grad-accum 4 -> effective batch 16, to fit the T4's
#     16 GB of memory (the A10G in trainer.py has 24 GB and uses batch 8).
#   * fp16/bf16 chosen automatically above (T4 -> fp16).
args = TrainingArguments(
    output_dir="rtdetrv2-sportsmot",
    num_train_epochs=20,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=1e-4,
    max_grad_norm=0.1,
    warmup_steps=300,
    lr_scheduler_type="cosine",
    bf16=USE_BF16,
    fp16=USE_FP16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,        # keep the epoch with the lowest eval loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    log_level="info",                   # verbose Trainer logs for debugging
    logging_first_step=True,
    logging_steps=10,                   # log loss/lr every 10 steps
    disable_tqdm=False,                 # keep progress bars
    dataloader_num_workers=2,
    remove_unused_columns=False,        # the model needs our custom "labels" field
    report_to="none",
    seed=SEED,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

# Wrap training so any failure prints a full traceback + GPU memory state to the
# kernel log (invaluable when debugging a headless run you can't step through).
logger.info("Starting training: %d train / %d val images", len(train_samples), len(val_samples))
try:
    trainer.train()
except Exception:
    logger.exception("Training FAILED with an exception:")
    if torch.cuda.is_available():
        logger.error("CUDA memory summary:\n%s", torch.cuda.memory_summary())
    raise
logger.info("Training finished.")

In [ ]:
# --- 9. Evaluate on the held-out basketball sequence -----------------------
# eval_loss guided model selection; here we report COCO mAP for an interpretable
# quality number on real basketball frames.
print("Computing mAP on held-out basketball sequence...")
device = model.device
model.eval()
metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
with torch.no_grad():
    for img_path, boxes in val_samples:
        image = Image.open(img_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        outputs = model(**inputs)
        (result,) = processor.post_process_object_detection(
            outputs,
            target_sizes=torch.tensor([image.size[::-1]]).to(device),
            threshold=0.01,             # low threshold so mAP integrates full PR curve
        )
        gt_xyxy = torch.tensor(
            [[x, y, x + w, y + h] for x, y, w, h in boxes], dtype=torch.float32
        )
        metric.update(
            [{k: result[k].cpu() for k in ("boxes", "scores", "labels")}],
            [{"boxes": gt_xyxy, "labels": torch.zeros(len(boxes), dtype=torch.long)}],
        )
metrics = {k: float(v) for k, v in metric.compute().items() if v.numel() == 1}
print("Validation metrics:", metrics)

In [ ]:
# --- 10. Publish the model -------------------------------------------------
# With a token (interactive run): push model + processor + card to the Hub.
# Without a token (headless CLI run): save them to /kaggle/working/model,
# downloadable from the kernel's Output tab.
HAS_TOKEN = bool(os.environ.get("HF_TOKEN"))
OUT_DIR = "/kaggle/working/model"

if HAS_TOKEN:
    print(f"Pushing to hub: {HUB_MODEL_ID}")
    model.push_to_hub(HUB_MODEL_ID, private=False)
    processor.push_to_hub(HUB_MODEL_ID)
else:
    print(f"No token; saving model to {OUT_DIR}")
    model.save_pretrained(OUT_DIR)
    processor.save_pretrained(OUT_DIR)

card = f"""---
license: apache-2.0
base_model: {CHECKPOINT}
datasets:
- {DATASET_ID}
pipeline_tag: object-detection
tags:
- rt-detr-v2
- sports
- basketball
- player-detection
- tracking
---

# RT-DETRv2 (r50vd) fine-tuned on SportsMOT for player detection

Single-class (`player`) detector fine-tuned from `{CHECKPOINT}` on the annotated
train sequences of [{DATASET_ID}](https://huggingface.co/datasets/{DATASET_ID})
(basketball, soccer, and volleyball broadcast clips in MOTChallenge format).
Intended as the detection stage of a tracking-by-detection pipeline (e.g. with
ByteTrack via the `supervision` library) for basketball player tracking.

## Validation (held-out basketball sequence `{VAL_SEQ}`)

| metric | value |
|---|---|
| mAP@[.5:.95] | {metrics.get('map', float('nan')):.4f} |
| mAP@50 | {metrics.get('map_50', float('nan')):.4f} |
| mAP@75 | {metrics.get('map_75', float('nan')):.4f} |
| mAR@100 | {metrics.get('mar_100', float('nan')):.4f} |

## Usage

```python
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForObjectDetection

processor = AutoImageProcessor.from_pretrained("{HUB_MODEL_ID}")
model = AutoModelForObjectDetection.from_pretrained("{HUB_MODEL_ID}")

image = Image.open("frame.jpg")
inputs = processor(images=image, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
results = processor.post_process_object_detection(
    outputs, target_sizes=torch.tensor([image.size[::-1]]), threshold=0.5
)
```

Trained on Kaggle (T4) from {len(train_samples)} frames (stride {TRAIN_STRIDE}) of 9
sequences, 20 epochs, 640x640, lr 5e-5 cosine, augmentation (hflip, color jitter, noise).
"""

if HAS_TOKEN:
    HfApi().upload_file(
        path_or_fileobj=card.encode(),
        path_in_repo="README.md",
        repo_id=HUB_MODEL_ID,
    )
    print("Done. Model at https://huggingface.co/" + HUB_MODEL_ID)
else:
    Path(OUT_DIR, "README.md").write_text(card)
    print(f"Done. Model + card saved to {OUT_DIR} - download it from the kernel Output tab")